In [ ]:
from qiskit import QuantumCircuit , QuantumRegister ,ClassicalRegister

In [ ]:
# f(x) values
# change only the right side values and run all cells again
# input order is q[2]q[1]q[0]
fx = {
    '000': 0,
    '001': 0,
    '010': 0,
    '011': 0,
    '100': 0,
    '101': 0,
    '110': 0,
    '111': 0
}

# constant: all 0 or all 1
# balanced: four 0 values and four 1 values

In [ ]:
# check the function before building Uf
inputs = [format(i,'03b') for i in range(8)]

if set(fx) != set(inputs):
    raise ValueError("give f(x) for all inputs from 000 to 111")

for i in fx:
    if fx[i] not in [0,1]:
        raise ValueError("DJ output should be 0 or 1")

if sum(fx.values()) not in [0,4,8]:
    raise ValueError("function should be constant or balanced")

In [ ]:
q = QuantumRegister(4,'q')
c = ClassicalRegister(3,'c')

In [ ]:
qc = QuantumCircuit(q,c)

In [ ]:
#input preparing
# q[0], q[1], q[2] are input qubits
# q[3] is the output qubit
qc.x(q[3])

qc.h(q[0])
qc.h(q[1])
qc.h(q[2])
qc.h(q[3])

qc.barrier()

In [ ]:
#oracle
# Uf |x>|y> = |x>|y xor f(x)>
# build the oracle from the table above
uf = QuantumCircuit(4, name='Uf')

for i in fx:
    if fx[i] == 1:
        # reverse the string so position 0 refers to q[0]
        bits = i[::-1]

        # change zero controls to one controls for this row
        for j in range(3):
            if bits[j] == '0':
                uf.x(j)

        # flip output only when the input matches this row
        uf.mcx([0,1,2],3)

        # restore the input qubits
        for j in range(3):
            if bits[j] == '0':
                uf.x(j)

uf.draw('mpl')

In [ ]:
# apply Uf to our circuit
qc.compose(uf, inplace=True)
qc.barrier()
qc.draw('mpl')

In [ ]:
qc.h(q[0])
qc.h(q[1])
qc.h(q[2])

qc.barrier()
qc.draw('mpl')

In [ ]:
qc.measure(q[0],c[0])
qc.measure(q[1],c[1])
qc.measure(q[2],c[2])

qc.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

# Create simulator
simulator = AerSimulator()

# Convert the oracle gates into gates supported by the simulator
compiled_qc = transpile(qc, simulator)
job = simulator.run(compiled_qc, shots=1000)

# Get result
result = job.result()
# Get counts
counts = result.get_counts()

print(counts)

In [ ]:
# for an ideal simulator and a promised constant or balanced oracle
if counts.get('000',0) == 1000:
    print("constant function")
elif counts.get('000',0) == 0:
    print("balanced function")
else:
    print("unexpected result")

for i in counts:
    print(i,"probability: ",counts[i]/1000)